# Word2Vec Skip-gram Pedagógico

Este notebook muestra paso a paso cómo entrenar embeddings Skip-gram para predecir palabras de contexto dado un target. Cada bloque está separado y tiene comentarios didácticos. Se imprime el avance en cada etapa relevante.

In [ ]:
# --- DEPENDENCIAS ---
import numpy as np  # Cálculo numérico
import matplotlib.pyplot as plt  # Visualización
from collections import Counter  # Contar elementos
from sklearn.metrics.pairwise import cosine_similarity  # Similitud coseno

In [ ]:
# --- 1. Corpus pequeño y simple ---
corpus = [
    "el gato come pescado fresco",
    "el perro come carne",
    "el gato duerme mucho",
    "el perro duerme poco",
    "el pescado es fresco",
    "la carne es sabrosa"
]
print("\n[OBJETIVO] Entrenar embeddings Skip-gram para predecir palabras de contexto dado un target.\n")

In [ ]:
# --- 2. Tokenización y vocabulario ---
# Separamos palabras y creamos diccionarios de índices
oraciones = [frase.split() for frase in corpus]
palabras = [palabra for oracion in oraciones for palabra in oracion]
vocabulario = sorted(set(palabras))
palabra_a_indice = {palabra: i for i, palabra in enumerate(vocabulario)}
indice_a_palabra = {i: palabra for palabra, i in palabra_a_indice.items()}
V = len(vocabulario)
print(f"Vocabulario: {vocabulario}\n")

In [ ]:
# --- 3. Generar pares skip-gram (target, contexto) ---
VENTANA = 2  # Palabras a izquierda y derecha
pares = []
for oracion in oraciones:
    indices = [palabra_a_indice[w] for w in oracion]
    for i, objetivo in enumerate(indices):
        for j in range(max(0, i-VENTANA), min(len(indices), i+VENTANA+1)):
            if i != j:
                pares.append((objetivo, indices[j]))
print(f"Ejemplo de pares (objetivo -> contexto):")
for t, c in pares[:5]:
    print(f"  '{indice_a_palabra[t]}' -> '{indice_a_palabra[c]}'")
print()

In [ ]:
# --- 4. Inicializar embeddings ---
DIM_EMB = 2  # Dimensión 2 para graficar
W_objetivo = np.random.normal(0, 0.1, (V, DIM_EMB))  # Embeddings de palabras objetivo
W_contexto = np.random.normal(0, 0.1, (V, DIM_EMB))  # Embeddings de contexto
W_objetivo_ini = W_objetivo.copy()  # Para comparar antes/después

In [ ]:
# --- 5. Ejemplo de forward pass antes de entrenar ---
print("[EJEMPLO] Forward pass antes de entrenar:")
ejemplo_t, ejemplo_c = pares[0]
v_t = W_objetivo[ejemplo_t]
scores = W_contexto @ v_t  # Producto punto con todos los contextos
probs = np.exp(scores) / np.exp(scores).sum()  # Softmax
print(f"Objetivo: '{indice_a_palabra[ejemplo_t]}' | Contexto real: '{indice_a_palabra[ejemplo_c]}'")
print("Probabilidades de contexto predichas:")
for idx, p in enumerate(probs):
    print(f"  {indice_a_palabra[idx]:10s}: {p:.3f}")
loss = -np.log(probs[ejemplo_c])
print(f"Cross-entropy loss para el par: {loss:.4f}\n")

In [ ]:
# --- 6. Entrenamiento Skip-gram ---
TASA_APRENDIZAJE = 0.1
EPOCAS = 100
print("[ENTRENAMIENTO]\n")
for epoca in range(EPOCAS):
    np.random.shuffle(pares)
    perdida_total = 0
    for t, c in pares:
        # FORWARD: calcular predicción
        v_t = W_objetivo[t]
        scores = W_contexto @ v_t
        probs = np.exp(scores) / np.exp(scores).sum()
        # LOSS: calcular pérdida
        perdida = -np.log(probs[c])
        perdida_total += perdida
        # BACKWARD: gradiente
        grad_salida = probs.copy()
        grad_salida[c] -= 1
        # UPDATE: ajustar embeddings
        W_contexto -= TASA_APRENDIZAJE * np.outer(grad_salida, v_t)
        W_objetivo[t] -= TASA_APRENDIZAJE * (W_contexto.T @ grad_salida)
    if (epoca+1) % 20 == 0:
        print(f"Época {epoca+1:3d} | Pérdida promedio: {perdida_total/len(pares):.4f}")

In [ ]:
# --- 7. Visualización de embeddings antes y después ---
plt.figure(figsize=(10,5))
for i, (mat, titulo) in enumerate(zip([W_objetivo_ini, W_objetivo], ["Antes de entrenar", "Después de entrenar"])):
    plt.subplot(1,2,i+1)
    plt.scatter(mat[:,0], mat[:,1], color='steelblue')
    for idx, palabra in indice_a_palabra.items():
        plt.text(mat[idx,0], mat[idx,1], palabra, fontsize=12)
    plt.title(titulo)
    plt.axis('equal')
plt.suptitle("Embeddings Skip-gram — Convergencia Visual")
plt.tight_layout()
plt.show()

In [ ]:
# --- 8. Similitud coseno entre palabras ---
print("\n[SIMILITUD COSENO ENTRE PALABRAS]")
emb_norm = W_objetivo / (np.linalg.norm(W_objetivo, axis=1, keepdims=True) + 1e-8)  # Normalizar
matriz_sim = cosine_similarity(emb_norm)
for idx, palabra in indice_a_palabra.items():
    sims = matriz_sim[idx]
    top_idx = np.argsort(sims)[::-1][1:4]  # top 3 (excluye sí mismo)
    vecinos = [(indice_a_palabra[i], sims[i]) for i in top_idx]
    vecinos_str = ", ".join(f"{v} ({s:.2f})" for v, s in vecinos)
    print(f"  {palabra:10s} → {vecinos_str}")

In [ ]:
# --- 9. Mini test final: palabras más similares a "perro" ---
print("\n[TEST FINAL] Palabras más similares a 'perro':")
if "perro" in palabra_a_indice:
    idx = palabra_a_indice["perro"]
    sims = matriz_sim[idx]
    top_idx = np.argsort(sims)[::-1][1:4]
    for i in top_idx:
        print(f"  {indice_a_palabra[i]} ({sims[i]:.2f})")
else:
    print("  'perro' no está en el vocabulario.")